# CosyVoice 2 Audio Tokenizer Example

This notebook demonstrates the CosyVoice 2 speech tokenizer that extracts semantic tokens at 25 Hz frame rate.

**Key Properties:**
- Codebook size: 6561 tokens (81²)
- Token rate: 25 Hz
- Input sample rate: 16 kHz
- Output sample rate: 24 kHz
- Architecture: Encoder + Flow Model + Vocoder for decoding

## Setup

In [1]:
import sys
import torch
import numpy as np
from datasets import load_dataset
from IPython.display import Audio, display
import matplotlib.pyplot as plt

# Add our tokenizer module to path
sys.path.append('../src')

# Import our tokenizer registry
from audio_tokenizers import get_tokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.10.0.dev20251108+cu126
CUDA available: True
Using device: cuda


## Load ESB Diagnostic AMI Dataset

In [2]:
# Load the AMI clean split
print("Loading ESB diagnostic dataset - AMI clean split...")
dataset = load_dataset('esb/diagnostic-dataset', 'ami', split='clean')

print(f"Dataset loaded with {len(dataset)} samples")
print(f"Sample keys: {dataset[0].keys()}")

Loading ESB diagnostic dataset - AMI clean split...
Dataset loaded with 770 samples
Sample keys: dict_keys(['audio', 'ortho_transcript', 'norm_transcript', 'id', 'dataset'])


## Find Long Audio Samples

In [3]:
# Calculate duration for each sample and find the longest ones
durations = []
for i, sample in enumerate(dataset):
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    duration = len(audio_array) / sr
    durations.append((i, duration, sample['norm_transcript']))

# Sort by duration and get top 5 longest
durations.sort(key=lambda x: x[1], reverse=True)
long_samples = durations[:5]

print("Top 5 longest audio samples:")
for idx, (i, dur, transcript) in enumerate(long_samples):
    print(f"{idx+1}. Sample {i}: {dur:.2f} seconds")
    print(f"   Transcript: {transcript[:100]}..." if len(transcript) > 100 else f"   Transcript: {transcript}")
    print()

Top 5 longest audio samples:
1. Sample 85: 11.77 seconds
   Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c ...

2. Sample 1: 11.70 seconds
   Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would lik...

3. Sample 100: 11.69 seconds
   Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go ...

4. Sample 31: 11.44 seconds
   Transcript: and also we talked about um a location function where maybe you could press a button on the t v and ...

5. Sample 210: 11.33 seconds
   Transcript: the the problem is if you have to go across the building and it adds some overhead every time you wa...



## Initialize CosyVoice 2 Tokenizer

In [4]:
# Initialize the tokenizer
print("Loading CosyVoice 2 tokenizer...")
tokenizer = get_tokenizer('cosyvoice2', device=device)

print(f"Tokenizer loaded: {tokenizer}")
print(f"  Input sample rate: {tokenizer.sample_rate} Hz")
print(f"  Output sample rate: {tokenizer.output_sample_rate} Hz")
print(f"  Codebook size: {tokenizer.codebook_size:,}")
print(f"  Downsample rate: {tokenizer.downsample_rate}x")
print(f"  Token rate: 25 Hz (fixed for CosyVoice 2)")

Loading CosyVoice 2 tokenizer...


2025-11-09 14:58:34,409 - modelscope - INFO - Target directory already exists, skipping creation.
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:69: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Model directory: /users/xyixuan/.cache/modelscope/hub/models/iic/CosyVoice2-0___5B
Loading tokenizer from: /users/xyixuan/.cache/modelscope/hub/models/iic/CosyVoice2-0___5B/speech_tokenizer_v2.onnx
Loading CosyVoice 2 decoder models...
Loading config from: /users/xyixuan/.cache/modelscope/hub/models/iic/CosyVoice2-0___5B/cosyvoice2.yaml


/usr/local/lib/python3.12/dist-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
2025-11-09 14:58:39,365 INFO input frame rate=25
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


✓ CosyVoice2 decoder models loaded successfully
✓ Decoder models loaded successfully
Tokenizer loaded: CosyVoice2Tokenizer(checkpoint='iic/CosyVoice2-0.5B', device='cuda', sample_rate=16000, output_sample_rate=24000, decoder_loaded=True)
  Input sample rate: 16000 Hz
  Output sample rate: 24000 Hz
  Codebook size: 6,561
  Downsample rate: 640x
  Token rate: 25 Hz (fixed for CosyVoice 2)


## Tokenize and Reconstruct Audio Samples

In [5]:
# Process the top 3 longest samples
results = []

for idx in range(min(3, len(long_samples))):
    sample_idx, duration, transcript = long_samples[idx]
    sample = dataset[sample_idx]
    
    print(f"\n{'='*60}")
    print(f"Processing Sample {idx+1} (index {sample_idx})")
    print(f"Duration: {duration:.2f} seconds")
    print(f"Transcript: {transcript[:150]}..." if len(transcript) > 150 else f"Transcript: {transcript}")
    
    # Get audio data
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    
    # Convert to tensor
    audio_tensor = torch.from_numpy(audio_array).float()
    
    # Tokenize
    print("\nTokenizing...")
    tokens, encode_info = tokenizer.encode(audio_tensor, sr=sr)
    
    # Show token information
    print(f"Token shape: {tokens.shape}")
    print(f"Number of tokens: {tokens.numel()}")
    print(f"Tokens per second: {tokens.numel() / duration:.1f}")
    
    # Show first 20 discrete token values
    token_values = tokens.squeeze().cpu().numpy()
    print(f"\nFirst 20 discrete token IDs:")
    print(token_values[:20])
    print(f"Token ID range: [{token_values.min()}, {token_values.max()}]")
    print(f"Unique tokens used: {len(np.unique(token_values))}")
    
    # Try to decode back to audio
    try:
        print("\nDecoding tokens back to audio...")
        reconstructed, decode_info = tokenizer.decode(tokens)
        
        print(f"Reconstructed shape: {reconstructed.shape}")
        print(f"Output sample rate: {decode_info['output_sample_rate']} Hz")
        
        # Store results
        results.append({
            'original': audio_array,
            'original_sr': sr,
            'reconstructed': reconstructed.squeeze().cpu().numpy(),
            'reconstructed_sr': decode_info['output_sample_rate'],
            'tokens': token_values,
            'transcript': transcript,
            'duration': duration
        })
    except Exception as e:
        print(f"⚠ Decoding not available: {e}")
        results.append({
            'original': audio_array,
            'original_sr': sr,
            'reconstructed': None,
            'reconstructed_sr': None,
            'tokens': token_values,
            'transcript': transcript,
            'duration': duration
        })
    
print(f"\n{'='*60}")
print("Processing complete!")


Processing Sample 1 (index 85)
Duration: 11.77 seconds
Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c i can see like some kind of a thing where like you...

Tokenizing...
Token shape: torch.Size([1, 295])
Number of tokens: 295
Tokens per second: 25.1

First 20 discrete token IDs:
[1598 4672 4916 2233   64   72 1678 1566 4831 3886 4177 2909 1449  168
 3003 2204 3994 3995 1833  186]
Token ID range: [0, 6534]
Unique tokens used: 246

Decoding tokens back to audio...


/iopsstor/scratch/cscs/xyixuan/benchmark-audio-tokenizer/examples/../src/repos/cosyvoice/cosyvoice/cli/model.py:286: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self.fp16):


Reconstructed shape: torch.Size([1, 1, 282240])
Output sample rate: 24000 Hz

Processing Sample 2 (index 1)
Duration: 11.70 seconds
Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would like to see in a convenient practical

Tokenizing...
Token shape: torch.Size([1, 293])
Number of tokens: 293
Tokens per second: 25.0

First 20 discrete token IDs:
[1814 5647 6374 5644 3465 4439 2216  146  470  389  722  698 5075 3509
 6414 4215 4218 3894 1707 3975]
Token ID range: [2, 6537]
Unique tokens used: 233

Decoding tokens back to audio...
Reconstructed shape: torch.Size([1, 1, 281280])
Output sample rate: 24000 Hz

Processing Sample 3 (index 100)
Duration: 11.69 seconds
Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go for that if we want

Tokenizing...
Token shape: torch.Size([1, 293])
Number of tokens: 293
Tokens per second: 25.1

First 20 discrete token IDs:
[3946 3960 4439   

## Play Original vs Reconstructed Audio

In [6]:
# Display audio players for comparison
for idx, result in enumerate(results):
    print(f"\n{'='*60}")
    print(f"Sample {idx+1} - Duration: {result['duration']:.2f}s")
    print(f"Transcript: {result['transcript'][:200]}..." if len(result['transcript']) > 200 else f"Transcript: {result['transcript']}")
    print(f"\nTokens used: {len(result['tokens'])} ({len(result['tokens'])/result['duration']:.1f} tokens/sec)")
    
    print("\nOriginal Audio (16 kHz):")
    display(Audio(result['original'], rate=result['original_sr']))
    
    if result['reconstructed'] is not None:
        print("\nReconstructed Audio (24 kHz):")
        display(Audio(result['reconstructed'], rate=result['reconstructed_sr']))
    else:
        print("\n⚠ Reconstruction not available (decoder models not loaded)")
    
    # Compression ratio
    original_size = len(result['original']) * 2  # 16-bit audio
    token_size = len(result['tokens']) * 2  # 13-bit tokens (log2(6561) ≈ 12.7)
    compression_ratio = original_size / token_size
    print(f"\nCompression ratio: {compression_ratio:.1f}x")


Sample 1 - Duration: 11.77s
Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c i can see like some kind of a thing where like you sort of have like the number come up on the t v l...

Tokens used: 295 (25.1 tokens/sec)

Original Audio (16 kHz):



Reconstructed Audio (24 kHz):



Compression ratio: 638.4x

Sample 2 - Duration: 11.70s
Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would like to see in a convenient practical

Tokens used: 293 (25.0 tokens/sec)

Original Audio (16 kHz):



Reconstructed Audio (24 kHz):



Compression ratio: 638.9x

Sample 3 - Duration: 11.69s
Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go for that if we want

Tokens used: 293 (25.1 tokens/sec)

Original Audio (16 kHz):



Reconstructed Audio (24 kHz):



Compression ratio: 638.4x


## Summary Statistics

In [7]:
# Calculate summary statistics
print("Summary of Tokenization Results:")
print("="*60)

total_duration = sum(r['duration'] for r in results)
total_tokens = sum(len(r['tokens']) for r in results)

print(f"Total audio processed: {total_duration:.2f} seconds")
print(f"Total tokens generated: {total_tokens:,}")
print(f"Average tokens per second: {total_tokens/total_duration:.1f}")
print(f"Average compression ratio: {640:.1f}x")

print(f"\nCosyVoice 2 Properties:")
print(f"  Bitrate: {25 * 13 / 1000:.2f} kbps (approx)")
print(f"  Token rate: 25 Hz")
print(f"  Bits per token: ~13 (log2(6561))")
print(f"  Codebook size: 6,561 (81²)")
print(f"  Input: 16 kHz mono")
print(f"  Output: 24 kHz mono (via Flow + Vocoder)")

Summary of Tokenization Results:
Total audio processed: 35.16 seconds
Total tokens generated: 881
Average tokens per second: 25.1
Average compression ratio: 640.0x

CosyVoice 2 Properties:
  Bitrate: 0.33 kbps (approx)
  Token rate: 25 Hz
  Bits per token: ~13 (log2(6561))
  Codebook size: 6,561 (81²)
  Input: 16 kHz mono
  Output: 24 kHz mono (via Flow + Vocoder)


## Summary

CosyVoice 2 tokenizer successfully:
- Encodes audio to semantic tokens at 25 Hz (fixed rate)
- Uses a codebook of 6,561 tokens (81²)
- Provides ~640x compression ratio
- Can decode tokens back to 24 kHz audio using Flow Model + Vocoder (when available)

The semantic tokens can be used for:
- Speech analysis and understanding
- Input to language models for speech generation  
- Speech editing and manipulation tasks
- Building blocks for speech synthesis pipelines